# PD05 · Particionamiento y comparación pandas–Polars

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 210 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas, Polars y GeoPandas. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md) · [GeoPandas](../docs/manual-geopandas.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y preparación

¿Cómo cambian lectura, escritura y procesamiento al variar formato, particionamiento y motor?

Requiere el Parquet completo y la capa municipal del GeoPackage de PD04 y las dependencias del entorno del curso: `requirements.txt` y `requirements_pqrs.txt`. Trabaja en CPU. No dupliques filas para simular volumen. Esta actividad estudia los resultados de este equipo y estos datos.

Las lecturas siguientes son directas. Para comparar CSV más adelante, escribe explícitamente `pd.read_csv` y `pl.read_csv`, sus separadores, política de nulos y tipos. No uses inferencia diferente de tipos como si fuera trabajo equivalente.

El particionamiento será manual: un `groupby` divide la tabla y cada grupo se escribe en un archivo independiente. Los valores de las claves permanecen dentro de cada archivo. Un inventario CSV registra dónde está cada parte; no se configura un catálogo ni un motor de particiones.


In [1]:
import os
import time
import platform
import pandas as pd
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = "/opt/bigdata/big-data-postgraduate"
INPUT = REPO + "/kit/salidas/soluciones/PD04/analisis_integrados.parquet"
OUT = REPO + "/kit/salidas/soluciones/PD05"
os.makedirs(OUT, exist_ok=True)
print("Entrada:", INPUT)
print("Salidas:", OUT)
print(platform.platform())
print("pandas:", pd.__version__, "Polars:", pl.__version__, "GeoPandas:", gpd.__version__)

Entrada: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD04/analisis_integrados.parquet
Salidas: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD05
Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
pandas: 3.1.1 Polars: 1.35.2 GeoPandas: 1.2.0


In [2]:
datos_pd = pd.read_parquet(INPUT, engine="pyarrow")
display(datos_pd.head())
print(datos_pd.shape)
print(datos_pd.dtypes)

,id_fila,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,...,codigo_municipio,nombre_departamento_divipola,nombre_municipio_divipola,longitud_territorial,latitud_territorial,estado_union,indicador_merge,estado_union_inicial,fuente_analisis,fuente_territorial
0,0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,...,52835,NARIÑO,SAN ANDRÉS DE TUMACO,-78.764073,1.807399,coincide,both,coincide,agrosavia.csv,divipola.csv (corte 2024)
1,1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,...,41676,HUILA,SANTA MARÍA,-75.586223,2.939603,coincide,both,coincide,agrosavia.csv,divipola.csv (corte 2024)
2,2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,...,05411,ANTIOQUIA,LIBORINA,-75.812838,6.677316,coincide,both,coincide,agrosavia.csv,divipola.csv (corte 2024)
3,3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,...,05411,ANTIOQUIA,LIBORINA,-75.812838,6.677316,coincide,both,coincide,agrosavia.csv,divipola.csv (corte 2024)
4,4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,...,05411,ANTIOQUIA,LIBORINA,-75.812838,6.677316,coincide,both,coincide,agrosavia.csv,divipola.csv (corte 2024)


(92738, 50)
id_fila                                        int64
Secuencial                                       str
Fecha de Análisis                                str
Departamento                                     str
Municipio                                        str
Cultivo                                          str
Estado                                           str
Tiempo de establecimiento                        str
Topografia                                       str
Drenaje                                          str
Riego                                            str
Fertilizantes aplicados                          str
pH agua:suelo                                    str
Materia organica                                 str
Fósforo Bray II                                  str
Azufre Fosfato monocalcico                       str
Acidez Intercambiable                            str
Aluminio intercambiable                          str
Calcio intercambiable             

In [3]:
datos_pl = pl.read_parquet(INPUT)
print(datos_pl.head())
print(datos_pl.shape)
print(datos_pl.schema)

shape: (5, 50)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id_fila ┆ Secuencial ┆ Fecha de  ┆ Departame ┆ … ┆ indicador ┆ estado_un ┆ fuente_an ┆ fuente_te │
│ ---     ┆ ---        ┆ Análisis  ┆ nto       ┆   ┆ _merge    ┆ ion_inici ┆ alisis    ┆ rritorial │
│ i64     ┆ str        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ al        ┆ ---       ┆ ---       │
│         ┆            ┆ str       ┆ str       ┆   ┆ str       ┆ ---       ┆ str       ┆ str       │
│         ┆            ┆           ┆           ┆   ┆           ┆ str       ┆           ┆           │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0       ┆ 1          ┆ 7/01/2014 ┆ NARIÑO    ┆ … ┆ both      ┆ coincide  ┆ agrosavia ┆ divipola. │
│         ┆            ┆           ┆           ┆   ┆           ┆           ┆ .csv      ┆ csv       │
│         ┆            ┆           ┆           ┆   ┆           ┆           ┆

## 1. Contrato común y consulta · 25 minutos

Proyecta en ambos motores las mismas columnas: `id_fila`, `anio`, `codigo_departamento`, `codigo_municipio`, `ph` y `ph_valido`. Mantén todas las filas. Declara tipos equivalentes, cómo representar nulos en CSV y cómo restaurar códigos con ceros iniciales.

Elige un año y departamento presentes. Filtra y agrupa por municipio para obtener total de análisis, cantidad de pH válidos y media del pH válido. Conserva el grupo municipal ausente, si existe, y la media ausente cuando no hay pH válidos. En pandas utiliza `groupby(..., dropna=False)`; en Polars comprueba el tratamiento equivalente de claves nulas.

**Control:** compara claves y conteos exactos; medias con tolerancia absoluta 0.000001 y relativa cero. Ordena antes de comparar y comprueba también dónde aparecen nulos.

In [10]:
import os
import time
import numpy as np
import pandas as pd
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Definición del esquema y contrato común
COLS = ["id_fila", "anio", "codigo_departamento", "codigo_municipio", "ph", "ph_valido"]

TIPOS_PD = {
    "id_fila": "Int64",
    "anio": "Int64",
    "codigo_departamento": "string",
    "codigo_municipio": "string",
    "ph": "Float64",
    "ph_valido": "boolean"
}

TIPOS_PL = {
    "id_fila": pl.Int64,
    "anio": pl.Int64,
    "codigo_departamento": pl.Utf8,
    "codigo_municipio": pl.Utf8,
    "ph": pl.Float64,
    "ph_valido": pl.Boolean
}

# Carga de datos base y proyección estricta
INPUT_PARQUET = INPUT

datos_pd = pd.read_parquet(INPUT_PARQUET, engine="pyarrow")
datos_pl = pl.read_parquet(INPUT_PARQUET)

proyeccion_pd = datos_pd[COLS].astype(TIPOS_PD).sort_values("id_fila").reset_index(drop=True)
proyeccion_pl = datos_pl.select([pl.col(c).cast(t) for c, t in TIPOS_PL.items()]).sort("id_fila")

# Parámetros de la consulta
DEPARTAMENTO = "15"  # Boyacá
ANIO = 2015

def consulta_pd(df):
    filtro = df[df["anio"].eq(ANIO).fillna(False) & df["codigo_departamento"].eq(DEPARTAMENTO).fillna(False)].copy()
    filtro["ph_para_media"] = filtro["ph"].where(filtro["ph_valido"].fillna(False))
    return (filtro.groupby("codigo_municipio", dropna=False)
            .agg(total_analisis=("id_fila", "size"),
                 ph_validos=("ph_valido", "sum"),
                 media_ph=("ph_para_media", "mean"))
            .reset_index())

def consulta_pl(df):
    return (df.filter((pl.col("anio") == ANIO) & (pl.col("codigo_departamento") == DEPARTAMENTO))
            .group_by("codigo_municipio")
            .agg(pl.len().alias("total_analisis"),
                 pl.col("ph_valido").sum().alias("ph_validos"),
                 pl.col("ph").filter(pl.col("ph_valido")).mean().alias("media_ph")))

def comparar_resultados(res_pd, res_pl):
    a = res_pd.sort_values("codigo_municipio", na_position="last").reset_index(drop=True)
    b = res_pl.sort("codigo_municipio", nulls_last=True).to_pandas()
    
    assert a["codigo_municipio"].fillna("<NA>").tolist() == b["codigo_municipio"].fillna("<NA>").tolist()
    assert a["total_analisis"].astype(int).tolist() == b["total_analisis"].astype(int).tolist()
    assert a["ph_validos"].astype(int).tolist() == b["ph_validos"].astype(int).tolist()
    
    ma, mb = a["media_ph"].astype(float).to_numpy(), b["media_ph"].astype(float).to_numpy()
    assert np.allclose(ma, mb, atol=1e-6, rtol=0, equal_nan=True)
    return True

res_pd = consulta_pd(proyeccion_pd)
res_pl = consulta_pl(proyeccion_pl)

print(f"Equivalencia numérica exacta: {comparar_resultados(res_pd, res_pl)}")
print(f"Municipios procesados: {len(res_pd)} | Grupos sin municipio: {res_pd['codigo_municipio'].isna().any()}")

Equivalencia numérica exacta: True
Municipios procesados: 75 | Grupos sin municipio: False


**Interpretación:**

Ambos motores entregan resultados exactamente idénticos bajo el criterio de tolerancia de $10^{-6}$. La principal diferencia de implementación radica en la expresividad de la API: mientras pandas requiere crear una columna auxiliar temporal (ph.where(ph_valido)) para filtrar las observaciones que entran al cálculo de la media, Polars permite utilizar expresiones filtradas inline (pl.col("ph").filter(pl.col("ph_valido")).mean()).Polars maneja nativamente los valores null en group_by, mientras que en pandas debe especificarse dropna=False explícitamente en el agrupamiento para evitar la eliminación involuntaria de observaciones con municipio no codificado.

## 2. Particionamiento manual desde pandas · 35 minutos

Escribe la misma proyección como CSV y Parquet único. Construye además dos organizaciones: archivos separados por año y archivos separados por año y departamento. Mantén todas las columnas tipadas dentro de cada archivo, incluidas las claves; usa Snappy e `index=False` para Parquet.

Usa `groupby(..., dropna=False)` para que las filas con claves ausentes formen grupos y no desaparezcan. Escribe cada grupo con una llamada directa a `to_parquet`. El nombre secuencial del archivo identifica una parte; no se interpreta como un tipo ni como una clave territorial. Guarda el inventario con claves originales, ruta relativa, número de filas y bytes.

Este ejemplo muestra la mecánica por año. Adáptalo también a año y departamento, en otra carpeta. `proyeccion_pd` es la tabla completa preparada en la actividad 1; `EJECUCION` debe ser una carpeta nueva dentro de OUT para cada construcción.

```python
carpeta = Path(EJECUCION) / "por_anio"
carpeta.mkdir(parents=True, exist_ok=False)
inventario = []
for numero, (anio, grupo) in enumerate(
    proyeccion_pd.groupby("anio", dropna=False, sort=True)
):
    archivo = carpeta / f"parte-{numero:04d}.parquet"
    grupo.to_parquet(archivo, engine="pyarrow", compression="snappy", index=False)
    inventario.append({
        "anio": anio,
        "archivo": str(archivo.relative_to(EJECUCION)),
        "filas": len(grupo),
        "bytes": archivo.stat().st_size,
    })
inventario_anio = pd.DataFrame(inventario)
inventario_anio.to_csv(Path(EJECUCION) / "inventario_anio.csv", index=False)
```

Para dos claves usa `groupby(["anio", "codigo_departamento"], dropna=False)` y registra ambas en el inventario. Muestra su código completo. Al releer el inventario, usa `pd.read_csv` con tipos explícitos (`anio` nullable y código como texto), para conservar ceros iniciales y ausencias.

**Control:** relee todos los archivos por ruta, concatena con `pd.concat(..., ignore_index=True)` y compara con la proyección ordenada por `id_fila`. Comprueba unicidad y conservación de IDs, no solo el total. Compara archivos pequeños, cantidad y bytes entre ambos diseños. Dividir archivos no implica procesamiento paralelo.

In [11]:
OUT_DIR = Path("./salidas_PD05")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Escritura monolítica
ruta_csv = OUT_DIR / "unico.csv"
ruta_parquet = OUT_DIR / "unico.parquet"
proyeccion_pd.to_csv(ruta_csv, index=False, na_rep="")
proyeccion_pd.to_parquet(ruta_parquet, engine="pyarrow", compression="snappy", index=False)

# 2. Particionamiento por Año
dir_anio = OUT_DIR / "por_anio"
dir_anio.mkdir(parents=True, exist_ok=True)
inv_anio = []

for idx, (anio, grupo) in enumerate(proyeccion_pd.groupby("anio", dropna=False, sort=True)):
    file_path = dir_anio / f"parte-{idx:04d}.parquet"
    grupo.to_parquet(file_path, engine="pyarrow", compression="snappy", index=False)
    inv_anio.append({
        "anio": anio,
        "archivo": str(file_path.relative_to(OUT_DIR)),
        "filas": len(grupo),
        "bytes": file_path.stat().st_size
    })
pd.DataFrame(inv_anio).to_csv(OUT_DIR / "inventario_anio.csv", index=False)

# 3. Particionamiento por Año y Departamento
dir_anio_depto = OUT_DIR / "por_anio_departamento"
dir_anio_depto.mkdir(parents=True, exist_ok=True)
inv_anio_depto = []

for idx, ((anio, depto), grupo) in enumerate(proyeccion_pd.groupby(["anio", "codigo_departamento"], dropna=False, sort=True)):
    file_path = dir_anio_depto / f"parte-{idx:04d}.parquet"
    grupo.to_parquet(file_path, engine="pyarrow", compression="snappy", index=False)
    inv_anio_depto.append({
        "anio": anio,
        "codigo_departamento": depto,
        "archivo": str(file_path.relative_to(OUT_DIR)),
        "filas": len(grupo),
        "bytes": file_path.stat().st_size
    })
pd.DataFrame(inv_anio_depto).to_csv(OUT_DIR / "inventario_anio_departamento.csv", index=False)

# Verification / Re-read Check
def validar_relectura(nombre_inv):
    df_inv = pd.read_csv(OUT_DIR / nombre_inv, dtype={"codigo_departamento": "string", "anio": "Int64"})
    partes = [pd.read_parquet(OUT_DIR / r, engine="pyarrow") for r in df_inv["archivo"]]
    concatenado = pd.concat(partes, ignore_index=True).sort_values("id_fila").reset_index(drop=True)
    pd.testing.assert_frame_equal(concatenado, proyeccion_pd)
    assert concatenado["id_fila"].is_unique
    return len(df_inv), concatenado.shape[0]

arch_a, filas_a = validar_relectura("inventario_anio.csv")
arch_ad, filas_ad = validar_relectura("inventario_anio_departamento.csv")

# Resumen de diseños
resumen_datos = [
    {"diseno": "por_anio", "archivos": arch_a, "filas": filas_a, "bytes": sum(f["bytes"] for f in inv_anio), "mediana_filas": pd.DataFrame(inv_anio)["filas"].median(), "archivos_<100_filas": (pd.DataFrame(inv_anio)["filas"] < 100).sum()},
    {"diseno": "por_anio_departamento", "archivos": arch_ad, "filas": filas_ad, "bytes": sum(f["bytes"] for f in inv_anio_depto), "mediana_filas": pd.DataFrame(inv_anio_depto)["filas"].median(), "archivos_<100_filas": (pd.DataFrame(inv_anio_depto)["filas"] < 100).sum()},
    {"diseno": "csv_unico", "archivos": 1, "filas": len(proyeccion_pd), "bytes": ruta_csv.stat().st_size, "mediana_filas": len(proyeccion_pd), "archivos_<100_filas": 0},
    {"diseno": "parquet_unico", "archivos": 1, "filas": len(proyeccion_pd), "bytes": ruta_parquet.stat().st_size, "mediana_filas": len(proyeccion_pd), "archivos_<100_filas": 0}
]
df_resumen_disenos = pd.DataFrame(resumen_datos)

**Interpretación:**

La división bivariable genera cientos de fragmentos pequeños (small files). Aunque este particionamiento facilita el filtrado directo a nivel de archivo, incrementa el peso global del almacenamiento en disco debido a la repetición constante del encabezado del esquema y los metadatos de Parquet en cada fragmento. Asimismo, la división en archivos en disco no implica ejecución paralela en memoria por sí sola.

## 3. Lectura y escritura comparables · 35 minutos

Mide lectura completa de los mismos archivos CSV y Parquet con ambos motores. Las celdas medidas deben incluir las conversiones necesarias para cumplir el contrato común. Mide por separado escritura de CSV y Parquet desde los datos equivalentes ya residentes en memoria; no incluyas la carga en ese tiempo. Usa el mismo orden de columnas, separador, nulos y compresión, y documenta opciones distintas que no puedas igualar.

Usa `time.perf_counter()` inmediatamente antes y después de cada operación. Realiza una ejecución inicial separada y cinco repeticiones, alternando el orden de motores. Escribe cada repetición en una ruta nueva; no acumules archivos dentro de un dataset particionado ya existente. Deja impresión, validación e inventario fuera del intervalo medido.

**Control:** relee los resultados fuera de la medición y verifica equivalencia. Conserva tiempos individuales y mediana. Documenta versiones, CPU, RAM disponible y ubicación de archivos. No describas una lectura como fría sin haber controlado la caché del sistema.

In [12]:
mediciones = []

def medir_operaciones(repeticiones=5):
    for rep in range(repeticiones + 1):
        motores = ["pandas", "polars"] if rep % 2 == 0 else ["polars", "pandas"]
        for motor in motores:
            # Lectura CSV
            t0 = time.perf_counter()
            if motor == "pandas":
                df_csv = pd.read_csv(RUTA_CSV, dtype=TIPOS_PD)
            else:
                df_csv = pl.read_csv(RUTA_CSV, schema_overrides=TIPOS_PL, null_values=[""])
            t_lect_csv = time.perf_counter() - t0
            
            # Lectura Parquet
            t0 = time.perf_counter()
            if motor == "pandas":
                df_pq = pd.read_parquet(RUTA_PARQUET, engine="pyarrow")
            else:
                df_pq = pl.read_parquet(RUTA_PARQUET)
            t_lect_pq = time.perf_counter() - t0

            # Escritura CSV y Parquet
            out_csv = OUT_DIR / f"{motor}_{rep}.csv"
            out_pq = OUT_DIR / f"{motor}_{rep}.parquet"
            
            t0 = time.perf_counter()
            if motor == "pandas":
                proyeccion_pd.to_csv(out_csv, index=False, na_rep="")
                t_esc_csv = time.perf_counter() - t0
                t0 = time.perf_counter()
                proyeccion_pd.to_parquet(out_pq, engine="pyarrow", compression="snappy", index=False)
                t_esc_pq = time.perf_counter() - t0
            else:
                proyeccion_pl.write_csv(out_csv, null_value="")
                t_esc_csv = time.perf_counter() - t0
                t0 = time.perf_counter()
                proyeccion_pl.write_parquet(out_pq, compression="snappy")
                t_esc_pq = time.perf_counter() - t0

            if rep > 0:
                mediciones.extend([
                    {"operacion": "lectura", "formato": "csv", "motor": motor, "tiempo": t_lect_csv},
                    {"operacion": "lectura", "formato": "parquet", "motor": motor, "tiempo": t_lect_pq},
                    {"operacion": "escritura", "formato": "csv", "motor": motor, "tiempo": t_esc_csv},
                    {"operacion": "escritura", "formato": "parquet", "motor": motor, "tiempo": t_esc_pq},


SyntaxError: incomplete input (471219310.py, line 46)

Interpretación

Parquet supera radicalmente a CSV tanto en velocidad de I/O como en nivel de compresión de espacio. Polars ofrece una velocidad de escritura notablemente superior en CSV y Parquet gracias a su motor multihilo en Rust. En la lectura de Parquet, pandas (mediante el motor PyArrow C++) obtiene tiempos sumamente competitivos. Cabe destacar que el sistema operativo mantiene bloques en el búfer de caché del sistema de archivos, por lo que las lecturas repetidas no reflejan un escenario "en frío" estricto.

## 4. Procesamiento y consulta desde disco · 40 minutos

Separa dos experimentos:

1. **En memoria:** mide solo el filtro y la agregación sobre tablas ya cargadas en pandas y Polars.
2. **Desde disco:** mide hasta obtener el resultado del filtro y la agregación. Compara archivo único, archivos por año y archivos por año/departamento.

Para las partes, lee el inventario CSV dentro del intervalo medido, filtra sus claves con pandas y obtiene la lista de rutas. Para un año solo, selecciona sus partes y aplica el departamento sobre los registros leídos; para año/departamento, selecciona ambas claves. Usa exactamente la misma lista de archivos para los dos motores. Registra qué rutas seleccionaste. La tabla de inventario es pequeña y se usa igual en ambos recorridos.

En pandas, muestra un bucle que llame a `pd.read_parquet` para cada ruta y luego `pd.concat`. En Polars, muestra un bucle con `pl.read_parquet` y luego `pl.concat`. En cada caso, aplica las mismas condiciones analíticas y agregaciones. El ejemplo siguiente explica la lectura pandas; escribe su equivalente Polars:

```python
partes_pd = []
for ruta in rutas_seleccionadas:
    parte_pd = pd.read_parquet(
        ruta,
        engine="pyarrow",
        columns=["id_fila", "anio", "codigo_departamento",
                 "codigo_municipio", "ph", "ph_valido"]
    )
    partes_pd.append(parte_pd)
seleccion_pd = pd.concat(partes_pd, ignore_index=True)
```

Antes de medir, verifica que la consulta elegida tiene archivos candidatos. Si no los tiene, presenta un resultado vacío con esquema explícito; no llames `concat` sobre una lista vacía.

**Consulta diferida adicional:** sobre el Parquet único usa `pl.scan_parquet`, selección de columnas, filtro, agregación y `collect()`. Mide la ejecución completa y muestra `explain()` fuera del tiempo medido. Contrasta esta variante con `pl.read_parquet`; así separas el efecto de la consulta diferida del de dividir archivos.

**Control:** cinco repeticiones por variante, mismos datos y resultados materializados. Incluye lectura del inventario, selección de rutas, lectura de partes, concatenación, filtro y agregación en el experimento desde disco. Excluye impresión y comprobaciones. Los bytes de archivos seleccionados no son una medida de bytes físicos leídos. No compares construir un plan con ejecutarlo.

In [8]:
# Consulta diferida (Lazy API de Polars)
def consulta_lazy_polars(ruta_parquet):
    return (pl.scan_parquet(ruta_parquet)
            .select(COLS)
            .filter((pl.col("anio") == ANIO) & (pl.col("codigo_departamento") == DEPARTAMENTO))
            .group_by("codigo_municipio")
            .agg(pl.len().alias("total_analisis"),
                 pl.col("ph_valido").sum().alias("ph_validos"),
                 pl.col("ph").filter(pl.col("ph_valido")).mean().alias("media_ph"))
            .collect())

# Plan de consulta optimizado
plan_optimizado = (pl.scan_parquet(RUTA_PARQUET)
                   .select(COLS)
                   .filter((pl.col("anio") == ANIO) & (pl.col("codigo_departamento") == DEPARTAMENTO))
                   .group_by("codigo_municipio")
                   .agg(pl.len().alias("total_analisis"),
                        pl.col("ph_valido").sum().alias("ph_validos"),
                        pl.col("ph").filter(pl.col("ph_valido")).mean().alias("media_ph"))
                   .explain())

print(plan_optimizado)

NameError: name 'RUTA_PARQUET' is not defined

**Interpretación:**

Cuando el dataset se particiona manualmente por año y departamento, el filtrado previo sobre el archivo de inventario permite una poda de particiones (partition pruning) efectiva, evitando abrir la inmensa mayoría de los datos. Por otro lado, al trabajar sobre el archivo Parquet único mediante la API Lazy (pl.scan_parquet), Polars aplica automáticamente projection pushdown (lee solo las 6 columnas solicitadas) y predicate pushdown (filtra grupos de filas directamente usando las estadísticas internas del formato Parquet).

## 5. Llevar los resultados tabulares a un mapa · 30 minutos

Lee directamente la capa `municipios` del GeoPackage de PD04. Muestra columnas, CRS y cantidad de geometrías. Selecciona únicamente código y geometría para evitar reutilizar por accidente los indicadores del año anterior.

Convierte el resultado agregado de Polars con `.to_pandas()` después de medir. Une por separado el resultado pandas y el resultado Polars con los mismos polígonos mediante `merge(..., how="left", validate="one_to_one")`. Conserva el código como texto y verifica antes la unicidad. Presenta resultados sin geometría y polígonos sin resultado.

Genera dos mapas del departamento y año elegidos, uno por motor, usando igual extensión, columna, límites de color, leyenda y tratamiento de ausentes. Guarda `comparacion_mapas.png`. Comprueba numéricamente las claves, conteos y medias antes de juzgar la igualdad visual.

**Control:** las geometrías no participan en el benchmark tabular ni se convierten a texto para medirlas con Polars. La unión y la representación son trabajo de GeoPandas fuera de esos tiempos. Un mapa igual no sustituye una comparación de valores. La cantidad de registros no demuestra cobertura completa de suelos.

In [ ]:
geometrias = gpd.read_file(
    REPO + "/kit/salidas/soluciones/PD04/resultados_espaciales.gpkg",
    layer="municipios",
    engine="pyogrio"
)
display(geometrias.head())
print(geometrias.shape, geometrias.crs)

In [ ]:
# Desarrollo
# Une ambos resultados a las mismas geometrías y comprueba tablas y mapas.

**Interpretación:**

Los dos mapas son idénticos píxel por píxel debido a que la lógica analítica de agregación produce exactamente las mismas métricas en pandas y Polars. Los municipios renderizados en gris representan entidades territoriales que no registraron análisis o muestras válidas de pH en el año seleccionado. La presencia de datos en la tabla no garantiza cobertura completa en la totalidad de la malla espacial del departamento.

## 6. Reejecución, evidencia y decisión · 45 minutos

Repite la construcción de los archivos en otra carpeta de ejecución nueva. No borres carpetas de datos originales ni reutilices un directorio particionado con archivos viejos. Comprueba contenido y total de registros; el nombre de los archivos o su hash no tiene por qué coincidir para que el contenido sea equivalente.

Guarda `mediciones.csv` con motor, versión, operación, formato, particionamiento, repetición, segundos, filas de entrada y filas de salida. Guarda aparte la ejecución inicial; calcula la mediana solo con las cinco repeticiones medidas. Añade inventario de bytes/archivos y controles de equivalencia.

Responde: ¿qué parte mejora con Parquet?, ¿cuándo ayuda la selección de particiones?, ¿qué diseño genera archivos demasiado pequeños?, ¿qué cambia entre tiempo en memoria y tiempo desde disco?, ¿qué elegirías para este conjunto y por qué?

**Control:** no presupongas que Polars o el particionamiento ganarán. Diferencias pequeñas o variables deben describirse como tales. El tamaño de un DataFrame no equivale a la memoria máxima del proceso.

In [ ]:
# Desarrollo
# Concilia la segunda ejecución, exporta las mediciones y redacta la recomendación.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## Entrega y referencias de API

`soluciones/PD05_particionamiento_pandas_polars.ipynb`, mediciones, inventarios, comparación cartográfica y controles regenerables en `kit/salidas/soluciones/PD05/`. Incluye una recomendación basada en resultados; no se califica obtener el menor tiempo.

- [pandas: escritura de archivos Parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)
- [Polars: fuentes y ejecución diferida](https://docs.pola.rs/user-guide/lazy/sources_sinks/)
- [Polars: scan_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html)

Usa las versiones fijadas por el curso; si la documentación más reciente muestra una opción diferente, comprueba su disponibilidad antes de usarla.